# 06 — Pemët e vendimit dhe ansamblet

## Konteksti dhe problemi shkencor

Klasifikimi i fazave të materialit demonstron ndarjet, papastërtinë, overfitting, bagging dhe random forest.

### Objektivat e të nxënit

- të llogaritni Gini dhe fitimin;
- të kontrolloni thellësinë;
- të kuptoni bagging/random forest;
- të përdorni interpretueshmërinë me kujdes.

> **Parimi i laboratorit:** matematikë → implementim nga parimet e para → implementim me bibliotekë → krahasim numerik → interpretim fizik.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, Perceptron, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
DATA = Path("../datasets/synthetic")
SRC = Path("../src").resolve()
if str(SRC.parent) not in sys.path:
    sys.path.insert(0, str(SRC.parent))
np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


## 1. Faza materialesh

In [ ]:
df=pd.read_csv(DATA/'material_phases.csv'); X=df[['densitet_g_cm3','perçueshmëri','fortësi']]; y=df['fazë']; df.head()

## 2. Gini nga parimet e para

In [ ]:
from src.trees.manual import gini,split_gain
print('Gini root=',gini(y)); mask=X['fortësi']<4.5; print('gain=',split_gain(y,mask))

## 3. Kërkim manual i një ndarjeje

In [ ]:
candidates=np.quantile(X['fortësi'],np.linspace(.05,.95,40)); gains=[split_gain(y,X['fortësi']<t) for t in candidates]; best=candidates[np.argmax(gains)]; print(best,max(gains)); plt.plot(candidates,gains); plt.show()

## 4. Pemë scikit-learn dhe thellësia

In [ ]:
from sklearn.tree import DecisionTreeClassifier,plot_tree
for d in [1,2,4,None]:
 m=DecisionTreeClassifier(max_depth=d,random_state=0); print(d,cross_val_score(m,X,y,cv=5).mean())

In [ ]:
m=DecisionTreeClassifier(max_depth=3,random_state=0).fit(X,y); plt.figure(figsize=(14,7)); plot_tree(m,feature_names=X.columns,filled=False,fontsize=8); plt.show()

## 5. Bagging minimal dhe Random Forest

In [ ]:
from sklearn.ensemble import BaggingClassifier,RandomForestClassifier
bag=BaggingClassifier(estimator=DecisionTreeClassifier(),n_estimators=100,random_state=0); rf=RandomForestClassifier(n_estimators=250,random_state=0); print('tree',cross_val_score(DecisionTreeClassifier(random_state=0),X,y,cv=5).mean()); print('bag',cross_val_score(bag,X,y,cv=5).mean()); print('rf',cross_val_score(rf,X,y,cv=5).mean())

## 6. Feature importance dhe permutation importance

In [ ]:
from sklearn.inspection import permutation_importance
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.3,stratify=y,random_state=0); rf.fit(Xtr,ytr); imp=pd.Series(rf.feature_importances_,index=X.columns); perm=permutation_importance(rf,Xte,yte,n_repeats=20,random_state=0); pd.DataFrame({'impurity':imp,'permutation':perm.importances_mean},index=X.columns)

## 7. Boosting: vetëm ideja praktike

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
hgb=HistGradientBoostingClassifier(max_iter=100,random_state=0); print(cross_val_score(hgb,X,y,cv=5).mean())

## 8. Rast dështimi: pemë shumë e thellë

In [ ]:
tr=[]; te=[]; depths=range(1,16)
for d in depths:
 m=DecisionTreeClassifier(max_depth=d,random_state=0).fit(Xtr,ytr); tr.append(m.score(Xtr,ytr)); te.append(m.score(Xte,yte))
plt.plot(depths,tr,label='train'); plt.plot(depths,te,label='test'); plt.legend(); plt.show()

## 9. Ushtrime

- Llogaritni entropinë dhe krahasojeni me Gini.
- Ndërtoni manualisht një stump me një ndarje.
- Krahasoni `max_features` te random forest.
- Shtoni veçori zhurme dhe krahasoni dy llojet e importance.

## 10. Shembull i plotë

Krahasoni pemën e cekët, pemën e thellë dhe random forest me të njëjtat split-e; argumentoni kompromisin interpretueshmëri–performancë.

## 11. Playground

Ndryshoni rregullin sintetik që krijon fazat. Kur rregulli bëhet diagonal ose i lakuar, si ndryshon numri i ndarjeve të nevojshme?